In [ ]:
import scvelo as scv
from lets_plot import *
import polars as pl
import numpy as np
import cellestial as cl

LetsPlot.setup_html()

In [ ]:
adata = scv.datasets.pancreas()

In [ ]:
scv.pp.filter_genes(adata, min_shared_counts=20)
scv.pp.normalize_per_cell(adata)
scv.pp.filter_genes_dispersion(adata, n_top_genes=2000)
scv.pp.log1p(adata)
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

In [ ]:
scv.tl.velocity(adata)

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
scv.tools.velocity_embedding(adata, basis="umap")

In [ ]:
frame = (
    pl.from_numpy(adata.obsm["velocity_umap"], schema=["vel1", "vel2"])
    .with_columns(pl.from_numpy(adata.obsm["X_umap"], schema=["umap1", "umap2"]))
    .with_columns(pl.Series(adata.obs["clusters"]))
)

In [ ]:
frame = frame.with_columns(
    pl.col("vel1").add(pl.col("umap1")).alias("xend"),
    pl.col("vel2").add(pl.col("umap2")).alias("yend"),
)
frame

In [ ]:
base = (
    ggplot(frame)
    + geom_point(aes(x="umap1", y="umap2", color="clusters"), alpha=0.4)
    + scale_color_viridis(option="plasma")
    + ggsize(600, 400)
    + theme_void()
)
base

In [ ]:
base + geom_curve(
    aes("umap1", "umap2", xend="xend", yend="yend"),
    curvature=0.5,
    size=0.5,
    color="black",
    arrow=arrow(angle=20),
    # sampling=sampling_random(1000),
    ncp=100,
)+ggsize(1200,800)

In [ ]:
frame.schema

In [ ]:
# Compute angles and magnitudes using NumPy
angles = np.arctan2(frame["vel2"].to_numpy(), frame["vel1"].to_numpy())
magnitudes = np.sqrt(frame["vel1"].to_numpy()**2 + frame["vel2"].to_numpy()**2)

# Add back to the DataFrame
frame = frame.with_columns(
    pl.Series("angle", angles),
    pl.Series("magnitude", magnitudes)
)

# Bin angles to group similar directions
angle_bin_size = np.pi / 2  # Adjust bin size as needed
frame = frame.with_columns(
    (pl.col("angle") / angle_bin_size).floor().alias("angle_bin")
)

# Group by spatial bins
spatial_bin_size = 1  # Adjust for better merging
frame = frame.with_columns(
    (pl.col("umap1") / spatial_bin_size).floor().alias("x_bin"),
    (pl.col("umap2") / spatial_bin_size).floor().alias("y_bin")
)

In [ ]:
merged = (
    frame.group_by(["angle_bin", "x_bin", "y_bin"])
    .agg(
        pl.col("umap1").mean().alias("umap1"),  # Weighted centroid
        pl.col("umap2").mean().alias("umap2"),
        pl.col("vel1").sum().alias("vel1_sum"),  # Summing velocities instead of averaging
        pl.col("vel2").sum().alias("vel2_sum")
    )
    .with_columns(
        (pl.col("umap1") + pl.col("vel1_sum")).alias("xend"),  # Use integrated displacement
        (pl.col("umap2") + pl.col("vel2_sum")).alias("yend")
    )
)


In [ ]:
merged

In [ ]:

base + geom_curve(
    data=merged,
    mapping=aes("umap1", "umap2", xend="xend", yend="yend"),
    curvature=0,
    size=0.5,
    color="black",
    arrow=arrow(angle=10, type="closed"),
    sampling=sampling_random(1000),
    ncp=10
)


In [ ]:
scv.pl.velocity_embedding_stream(adata)

In [ ]:
adata

In [ ]:
cl.umap(adata,key="clusters", size=1,legend_ondata=True,ondata_size=8,axis_type="arrow")

In [ ]:
for key in adata.obsm:
    print(key)